# RGB (VisDrone DET) - 03 Testing

Que hace: elige un run ya entrenado, mide mAP en test, compara prediccion vs ground truth y procesa video (deteccion y tracking).

Que NO hace: no entrena (eso es `02_entrenar`).

## Parametros

`RUN = None` elige el run con mejor mAP50-95 que tenga `best.pt`. Para un run viejo (ej. `yolo11s_visdrone_tracking_opt-8`) apunta `OUT_DIR` a su carpeta y pon su nombre en `RUN`.

In [ ]:
DATASET_ZIP = "MyDrive/deteccion_drones/datasets/visdrone.zip"
DATASET_DIR = "visdrone"                        # subcarpeta en env.DATA_ROOT
DATA_YAML   = "VisDrone_Dataset/visdrone.yaml"  # relativo a DATASET_DIR
OUT_DIR     = "MyDrive/deteccion_drones/runs/rgb"

RUN   ="visdrone_person_yolo26s_1024_20260929-1747"   # nombre de la carpeta del run; None -> mejor mAP50-95
VIDEO = "MyDrive/deteccion_drones/videos/video1_testing.mp4"   # None para saltar la seccion de video


## Setup

Identica en todos los notebooks. El repo tiene que ser publico y `BRANCH` tiene que estar pusheada.

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "feat/framework"   # volver a "main" cuando se mergee

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

try:  # el IPython de Colab con Python 3.13 trae un autoreload que importa 'imp' (eliminado en 3.12)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload no disponible: reinicia el kernel tras editar src/")
from deteccion import env, data, viz
from deteccion import ExperimentConfig, Experiment, list_runs
env.mount_drive()


## 1. Obtener el dataset

Hace falta para medir mAP y mostrar ground truth. Idempotente (marcador `.done`).

In [ ]:
dataset_dir = env.DATA_ROOT / DATASET_DIR
env.unzip_once(env.drive_path(DATASET_ZIP), dataset_dir)
data_yaml = env.fix_data_yaml(dataset_dir / DATA_YAML)


## 2. Runs disponibles

mAP maximo de cada run sobre val, leido de su `results.csv`.

In [ ]:
runs = list_runs(OUT_DIR, sort="mAP50-95")
runs


## 3. Elegir run

El yaml guardado en el run apunta a una sesion vieja de Colab: se reemplaza por el de esta sesion.

In [ ]:
run_dir = runs[runs.has_best].path.iloc[0] if RUN is None else env.drive_path(OUT_DIR) / RUN
exp = Experiment.load(run_dir, data_yaml=str(data_yaml))
print(exp.run_dir)


## 4. mAP en test

**Solo al final.** Para comparar runs o decidir hiperparametros se usa val; si se elige mirando test, el numero de test deja de ser una estimacion honesta.

In [ ]:
exp.val("test")


## 5. Prediccion vs ground truth

Mismas imagenes cada vez (`seed`), para comparar runs a ojo.

In [ ]:
exp.show_pred_vs_gt(n=3, seed=0)


## 6. Video

Se guardan en `run_dir/videos/` (Drive). Tarda segun el largo del video; el tracking imprime avance cada 30 frames.

In [ ]:
if VIDEO:
    print(exp.predict_video(VIDEO))


In [ ]:
if VIDEO:
    print(exp.track(VIDEO))


## Conclusiones

_(completar: run elegido, mAP test, donde falla a ojo, calidad del tracking)_